# LTSR Colab Phase 9 — validate / archive

Phase 4–8完了runを検査し、raw runとgraphsのtar.gz＋SHA256をDriveへ保存する。

実行順は、Python 3.10確認 → Drive mount/source固定 → Phaseセルである。
Python環境導入でruntimeが再起動した場合は、再接続して最初のセルからやり直す。

In [ ]:
# 必ず最初に実行する。Colab UI kernelとは別に研究コード用Python 3.10を用意する。
import hashlib
import subprocess
import sys
import urllib.request
from pathlib import Path

PY310_ROOT = Path("/content/ltsr-py310")
PY310 = PY310_ROOT / "bin" / "python"
INSTALLER = Path("/content/Miniconda3-py310_23.11.0-2-Linux-x86_64.sh")
INSTALLER_URL = (
    "https://repo.anaconda.com/miniconda/"
    "Miniconda3-py310_23.11.0-2-Linux-x86_64.sh"
)
INSTALLER_SHA256 = (
    "35a58b8961e1187e7311b979968662c6223e86e1451191bed2e67a72b6bd0658"
)

def worker_version():
    if not PY310.is_file():
        return None
    return subprocess.check_output(
        [
            str(PY310), "-c",
            "import sys; print('.'.join(map(str, sys.version_info[:3])))",
        ],
        text=True,
    ).strip()

version = worker_version()
print("Colab controller:", sys.version)
print("LTSR worker before setup:", version)
if version is None or not version.startswith("3.10."):
    if not INSTALLER.is_file():
        urllib.request.urlretrieve(INSTALLER_URL, INSTALLER)
    digest = hashlib.sha256(INSTALLER.read_bytes()).hexdigest()
    if digest != INSTALLER_SHA256:
        raise RuntimeError(
            f"Miniconda installer checksum mismatch: {digest}"
        )
    subprocess.run(
        [
            "bash", str(INSTALLER), "-b", "-u",
            "-p", str(PY310_ROOT),
        ],
        check=True,
    )
    version = worker_version()
if version is None or not version.startswith("3.10."):
    raise RuntimeError(f"Python 3.10 worker setup failed: {version}")
print("LTSR worker Python 3.10: OK —", version)

In [ ]:
# Google認証とDrive mountはユーザー自身が行う。
from google.colab import drive
drive.mount("/content/drive")

import json
import os
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LTSR_colab")
REPO_ROOT = Path("/content/LTSR")
BRANCH = "20260726/gpu-scale-prep-colab"
PINNED_SOURCE_COMMIT = '8721989450bbf6170ffd62cf0ac82a866e800aed'
REPO_URL = (
    "https://github.com/blabo25226/"
    "Layer-selective_Transformer-based_Symbolic_Regression.git"
)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
lock_path = DRIVE_ROOT / "source_lock.json"

if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )

if PINNED_SOURCE_COMMIT:
    subprocess.run(
        ["git", "fetch", "origin", PINNED_SOURCE_COMMIT],
        cwd=REPO_ROOT,
        check=True,
    )
    locked_commit = PINNED_SOURCE_COMMIT
    subprocess.run(
        ["git", "checkout", "--detach", locked_commit],
        cwd=REPO_ROOT,
        check=True,
    )
    partial = lock_path.with_suffix(".json.partial")
    partial.write_text(
        json.dumps({"branch": BRANCH, "commit": locked_commit}, indent=2),
        encoding="utf-8",
    )
    os.replace(partial, lock_path)
elif lock_path.is_file():
    locked_commit = json.loads(lock_path.read_text(encoding="utf-8"))["commit"]
    subprocess.run(["git", "checkout", "--detach", locked_commit], cwd=REPO_ROOT, check=True)
else:
    locked_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()
    partial = lock_path.with_suffix(".json.partial")
    partial.write_text(
        json.dumps({"branch": BRANCH, "commit": locked_commit}, indent=2),
        encoding="utf-8",
    )
    os.replace(partial, lock_path)

sys.path.insert(0, str(REPO_ROOT / "src"))
from colab_runtime import assert_locked_source, require_python_310

require_python_310(PY310)
print("locked commit:", assert_locked_source(REPO_ROOT, DRIVE_ROOT))
print("Drive root:", DRIVE_ROOT)

In [ ]:
# /content is ephemeral, so every fresh Colab VM must restore worker packages.
import subprocess

dependency_probe = subprocess.run(
    [
        str(PY310), "-c",
        "import numpy, torch, pytorch_lightning, nesymres, pytest, pysr",
    ],
    cwd=REPO_ROOT,
)
if dependency_probe.returncode != 0:
    commands = [
        [str(PY310), "-m", "pip", "install", "--upgrade", "pip"],
        [
            str(PY310), "-m", "pip", "install", "torch==2.5.1",
            "--index-url", "https://download.pytorch.org/whl/cu124",
        ],
        [str(PY310), "-m", "pip", "install", "-r", "requirements/gpu.txt"],
        [str(PY310), "-m", "pip", "install", "-e", "NSRS/src"],
        [str(PY310), "-m", "pip", "install", "pytest", "pysr"],
    ]
    for command in commands:
        print("+", " ".join(command), flush=True)
        subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Python 3.10 worker dependencies: already installed")

In [ ]:
# Phase 4--8で同じ3値を使用する。別設定は必ず新しいRUN_IDにする。
from colab_runtime import config_for

RUN_KIND = "reduced"  # 承認済みの計算量削減版
RUN_ID = "colab_reduced_20260729_03"
MAX_PARALLEL_SEEDS = 2
CONFIG = config_for(
    RUN_KIND, RUN_ID, max_parallel_seeds=MAX_PARALLEL_SEEDS
)
print(json.dumps(CONFIG.scientific_dict(), indent=2, ensure_ascii=False))

In [ ]:
import subprocess

from colab_runtime import (
    assert_locked_source,
    restore_artifacts,
    restore_static_assets,
    sha256,
    sync_artifacts,
)

assert_locked_source(REPO_ROOT, DRIVE_ROOT)
restore_static_assets(REPO_ROOT, DRIVE_ROOT)
restore_artifacts(REPO_ROOT, DRIVE_ROOT, RUN_ID)
run_dir = REPO_ROOT / "results" / "runs" / RUN_ID
subprocess.run(
    [
        str(PY310),
        "scripts/aggregate_phase8_runs.py",
        "--run-dir",
        str(run_dir),
        "--seeds",
        *CONFIG.seeds.split(),
    ],
    cwd=REPO_ROOT,
    check=True,
)
phase8_summary_path = (
    run_dir / "phase8_lodo_multiseed" / "summary.json"
)
phase8_summary = json.loads(
    phase8_summary_path.read_text(encoding="utf-8")
)
if not phase8_summary.get("pysr_included"):
    raise RuntimeError(
        "Phase 8 aggregation did not include local PySR results"
    )
print("Phase 8 aggregation: PySR included")
subprocess.run(
    [str(PY310), "scripts/validate_gpu_run.py", "--run-dir", str(run_dir)],
    cwd=REPO_ROOT,
    check=True,
)
sync_artifacts(REPO_ROOT, DRIVE_ROOT, RUN_ID)

archive_dir = DRIVE_ROOT / "archives"
archive_dir.mkdir(parents=True, exist_ok=True)
graph_dir = REPO_ROOT / "graphs" / RUN_ID
if not graph_dir.is_dir():
    graph_dir.mkdir(parents=True, exist_ok=True)
    print("No graph artifacts were generated; archiving an empty graph directory")
archive = archive_dir / f"{RUN_ID}.tar.gz"
subprocess.run(
    [
        "tar", "-czf", str(archive),
        f"results/runs/{RUN_ID}", f"graphs/{RUN_ID}",
    ],
    cwd=REPO_ROOT,
    check=True,
)
digest = sha256(archive)
checksum = archive.with_suffix(archive.suffix + ".sha256")
checksum.write_text(f"{digest}  {archive.name}\n", encoding="utf-8")
print("archive:", archive)
print("sha256:", digest)